# Import libraries and dataset into environment

In [1]:
import dill
import os
import sys
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
import math as mt
from sklearn.metrics import confusion_matrix, roc_curve, auc
from MLstatkit import Bootstrapping
import json
import gc
import shap
from joblib import Parallel, delayed, externals
import multiprocessing
num_cores = multiprocessing.cpu_count() - 1
import defined_functions
from defined_functions import sum_metric, met_collate_func
defined_functions.pd = pd

In [2]:
path = os.getcwd()
sys.path.append(path)

save_file = os.path.join(path, "session.pkl")

with open(save_file, "rb") as f:
    state = dill.load(f)

split_list_valid_smote = state["split_list_valid_smote"]
split_list_valid_smote_final = state["split_list_valid_smote_final"]
split_list_fil_valid_smote_final = state["split_list_fil_valid_smote_final"]
split_list_sim_onset_valid_smote_final = state["split_list_sim_onset_valid_smote_final"]
split_list_vldiag_valid_smote_final = state["split_list_vldiag_valid_smote_final"]
split_list_diag1_valid_smote_final = state["split_list_diag1_valid_smote_final"]
split_list_diag2_valid_smote_final = state["split_list_diag2_valid_smote_final"]

# Define functions

In [3]:
# Define function to train Random Forest
def run_single_rf_split(i, split):
    """Process a single cross-vaidation data split loop iteration"""
    train_df = split['train'].copy()
    valid_df = split['valid'].copy()
    test_df = split['test'].copy()

    train_df['outcome'] = train_df['outcome'].cat.reorder_categories(["Non severe", "Severe"], ordered = False)
    valid_df['outcome'] = valid_df['outcome'].cat.reorder_categories(["Non severe", "Severe"], ordered = False)
    test_df['outcome'] = test_df['outcome'].cat.reorder_categories(["Non severe", "Severe"], ordered = False)

    train_x = train_df.drop(columns = ['outcome'])
    valid_x = valid_df.drop(columns = ['outcome'])
    test_x = test_df.drop(columns = ['outcome'])

    features = list(train_x.columns)
    cat_cols = ['age', 'gender', 'vaccination', 'comorbidity']
    for col in cat_cols:
        train_x[col] = train_x[col].astype('category')
        valid_x[col] = valid_x[col].astype('category')
        test_x[col] = test_x[col].astype('category')

    train_x = pd.get_dummies(train_x, columns = cat_cols, drop_first = False)
    valid_x = pd.get_dummies(valid_x, columns = cat_cols, drop_first = False)
    test_x = pd.get_dummies(test_x, columns = cat_cols, drop_first = False)

    feature_names = train_x.columns.tolist()

    # Force valid/test to have same columns and same order as train
    valid_x = valid_x.reindex(columns=feature_names, fill_value=0)
    test_x = test_x.reindex(columns=feature_names, fill_value=0)
    
    mapping = {"Non severe": 0, "Severe": 1}
    
    train_y = train_df['outcome'].map(mapping).astype(int)
    valid_y = valid_df['outcome'].map(mapping).astype(int)
    test_y = test_df['outcome'].map(mapping).astype(int)
    
    #pos = np.sum(train_y == 1)
    #neg = np.sum(train_y == 0)
    
    # Train the random forest model
    rf_model = RandomForestClassifier(
        n_estimators = 2500,    # Total number of decision trees to build
        max_depth = None,   # Maximum structural depth of each individual tree
        criterion = 'gini',    # Mathematical metric used to determine split quality
        min_samples_split = 55,   # Minimum number of samples required to split an internal structural node
        max_features = 'sqrt',    # Size of random subset of features ro inspect when looking for the best split
        min_samples_leaf = 25,    # Minimum number of samples required to be at a leaf node
        min_weight_fraction_leaf = 0.0,    # Minimum weighted fracion of the sum total of weights of all the input samples required to be at a leaf node
        max_leaf_nodes = 800,  # Maximum number of leaf nodes
        min_impurity_decrease = 0.0,   # A node will be split if this split induces a decrease of the impurity greater or equal to this value
        bootstrap = True,   # Boostrap sample when building trees
        oob_score = True,   # Whether to use out-of-bag samples to estimate the generalization score
        max_samples = 0.8, # Number of samples to draw from X to train each base estimator
    )
    rf = rf_model.fit(train_x, train_y)

    # Make prediction on Testing Set
    rf_pred_prob = rf.predict_proba(test_x)[:, 1]
    prediction = (rf_pred_prob >= 0.5).astype(int)
    prediction_labels = np.where(prediction == 0, "Non severe", "Severe")

    # Compute Confusion Matrix (Test)
    tn, fp, fn, tp = confusion_matrix(test_y, prediction, labels = [0, 1]).ravel()

    # Calculate Area Under the ROC curve
    fpr, tpr_curve, _ = roc_curve(test_y, rf_pred_prob, pos_label = 1)
    auc_val_rf = auc(fpr, tpr_curve)
    
    # Store standard structure dictionary to act like R's pROC container
    roc_container = {
        'actual': test_y,
        'probabilities': rf_pred_prob
    }
    
    # Calculate Area Under the Precision-Recall Curve (AUPRC / PR-AUC)
    auprc_val, prc_ci_lower, prc_ci_upper = Bootstrapping(test_y, rf_pred_prob, 'pr_auc')
    
    # Format the evaluation matrix lookip dataframe
    cfm_rf = pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index = ["Non severe", "Severe"],
        columns = ["Non severe", "Severe"]
    )
    cfm_rf.index.name = 'Prediction'
    cfm_rf.columns.name = 'Observed'

    # Calculate performance metrics
    accuracy_rf = (tp + tn) / (tn + fp + fn + tp) if (tn + fp + fn + tp) > 0 else 0
    sensitivity_rf = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity_rf = tn / (tn + fp) if (tn + fp) > 0 else 0
    npv_rf = tn / (tn + fn) if (tn + fn) > 0 else 0
    precision_rf = tp / (tp + fp) if (tp + fp) > 0 else 0

    # Make predictions on Training Set to gather accuracy
    rf_pred_train_prob = rf.predict_proba(train_x)[:, 1]
    rf_pred_train = (rf_pred_train_prob >= 0.5).astype(int)
    tn_tr, fp_tr, fn_tr, tp_tr = confusion_matrix(train_y, rf_pred_train, labels = [0, 1]).ravel()
    accuracy_rf_train = (tp_tr + tn_tr) / (tn_tr + fp_tr + fn_tr + tp_tr)
    
    explainer = shap.TreeExplainer(model = rf)
    shap_values = explainer(train_x).values
    
    return {
        "model": rf,
        "confusion_matrix": cfm_rf,
        "accuracy_training": accuracy_rf_train,
        "accuracy_testing": accuracy_rf,
        "sensitivity": sensitivity_rf,
        "specificity": specificity_rf,
        "precision": precision_rf,
        "npv": npv_rf,
        "roc": roc_container,
        "AUC_value": auc_val_rf,
        "PRC_val": auprc_val,
        "PRC_lower_ci": prc_ci_lower,
        "PRC_upper_ci": prc_ci_upper,
        "SHAP_values": shap_values,
        "prediction": prediction_labels,
        "pred_prob": rf_pred_prob
    }

# Define function to train Random Forest for 100 times in parallel
def model_func_rf_tune(data_list):
    """Main execution wrapper to distribute XGBoost tuning loops over system CPU threads."""
    # Count system resource availability profiles
    num_cores = multiprocessing.cpu_count() - 1
    
    if __name__ == '__main__':
        try:
            result_list = Parallel(n_jobs = num_cores)(
                delayed(run_single_rf_split)(i, data_list[i]) 
                for i in range(100)
                )
        finally:
            externals.loky.get_reusable_executor().shutdown(wait = True)
            gc.collect()

    return result_list

# Fitting dataset into the model

## Fitting data list without VL information

In [4]:
rf_fil_list = model_func_rf_tune(split_list_fil_valid_smote_final)
rf_fil_met_summary = sum_metric(rf_fil_list)
rf_fil_metrics_summary = rf_fil_met_summary["metric_summary"]
rf_fil_summary = met_collate_func(rf_fil_metrics_summary).assign(
    models = "Random Forest (No VL info & SMOTE)"
)
rf_fil_summary

Bootstrapping pr_auc: 100%|██████████| 1000/1000 [00:00<00:00, 1589.82it/s]


bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,37.9473,19.9196,61.9078,Random Forest (No VL info & SMOTE)
1,AUROC_value,92.7111,85.6441,97.6429,Random Forest (No VL info & SMOTE)
2,Accuracy,87.6828,79.9018,94.4184,Random Forest (No VL info & SMOTE)
3,Accuracy_train,88.4408,84.5145,92.0145,Random Forest (No VL info & SMOTE)
4,NPV,99.3575,98.6076,100.0000,Random Forest (No VL info & SMOTE)
5,Precision,18.3367,11.7246,31.0714,Random Forest (No VL info & SMOTE)
6,Sensitivity,81.5000,60.0000,100.0000,Random Forest (No VL info & SMOTE)
7,Specificity,87.8754,79.5872,95.0156,Random Forest (No VL info & SMOTE)


## Fitting data list with simulated VL at symptom onset

In [5]:
rf_vlsymp_sim_list = model_func_rf_tune(split_list_sim_onset_valid_smote_final)
rf_vlsymp_sim_met_summary = sum_metric(rf_vlsymp_sim_list)
rf_vlsymp_sim_metrics_summary = rf_vlsymp_sim_met_summary["metric_summary"]
rf_vlsymp_sim_summary = met_collate_func(rf_vlsymp_sim_metrics_summary).assign(
    models = "Random Forest (VL symp simulated & SMOTE)"
)
rf_vlsymp_sim_summary

Bootstrapping pr_auc: 100%|██████████| 1000/1000 [00:00<00:00, 1503.62it/s]


bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,37.6844,19.1849,63.6578,Random Forest (VL symp simulated & SMOTE)
1,AUROC_value,92.4702,84.7445,97.4264,Random Forest (VL symp simulated & SMOTE)
2,Accuracy,88.5076,81.5710,93.3535,Random Forest (VL symp simulated & SMOTE)
3,Accuracy_train,89.2586,86.0774,92.6571,Random Forest (VL symp simulated & SMOTE)
4,NPV,99.2790,98.6365,100.0000,Random Forest (VL symp simulated & SMOTE)
5,Precision,18.9007,11.7966,27.9545,Random Forest (VL symp simulated & SMOTE)
6,Sensitivity,79.1000,60.0000,100.0000,Random Forest (VL symp simulated & SMOTE)
7,Specificity,88.8006,80.9969,94.5561,Random Forest (VL symp simulated & SMOTE)


## Fitting data list with VL at diagnosis

In [6]:
rf_vldiag_list = model_func_rf_tune(split_list_vldiag_valid_smote_final)
rf_vldiag_met_summary = sum_metric(rf_vldiag_list)
rf_vldiag_metrics_summary = rf_vldiag_met_summary["metric_summary"]
rf_vldiag_summary = met_collate_func(rf_vldiag_metrics_summary).assign(
    models = "Random Forest (VL diag & SMOTE)"
)
rf_vldiag_summary

Bootstrapping pr_auc: 100%|██████████| 1000/1000 [00:00<00:00, 1445.68it/s]

Bootstrapping pr_auc: 100%|██████████| 1000/1000 [00:00<00:00, 1528.16it/s]


bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,42.6382,20.6410,67.2396,Random Forest (VL diag & SMOTE)
1,AUROC_value,93.5528,84.7850,98.1480,Random Forest (VL diag & SMOTE)
2,Accuracy,89.4894,83.0665,93.3535,Random Forest (VL diag & SMOTE)
3,Accuracy_train,89.7160,86.8614,92.9712,Random Forest (VL diag & SMOTE)
4,NPV,99.3577,98.6732,100.0000,Random Forest (VL diag & SMOTE)
5,Precision,20.6041,12.7484,29.1028,Random Forest (VL diag & SMOTE)
6,Sensitivity,81.2000,60.0000,100.0000,Random Forest (VL diag & SMOTE)
7,Specificity,89.7477,82.7025,94.0810,Random Forest (VL diag & SMOTE)


## Fitting data list with VL at diagnosis & VL at 1-day after diagnosis

In [7]:
rf_add1_list = model_func_rf_tune(split_list_diag1_valid_smote_final)
rf_add1_met_summary = sum_metric(rf_add1_list)
rf_add1_metrics_summary = rf_add1_met_summary["metric_summary"]
rf_add1_summary = met_collate_func(rf_add1_metrics_summary).assign(
    models = "Random Forest (VL diag + 1 & SMOTE)"
)
rf_add1_summary

Bootstrapping pr_auc: 100%|██████████| 1000/1000 [00:00<00:00, 1500.64it/s]


bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,44.3261,21.3254,69.4472,Random Forest (VL diag + 1 & SMOTE)
1,AUROC_value,93.8879,85.3224,98.5670,Random Forest (VL diag + 1 & SMOTE)
2,Accuracy,89.5257,84.5770,93.2100,Random Forest (VL diag + 1 & SMOTE)
3,Accuracy_train,90.6391,87.8453,93.4631,Random Forest (VL diag + 1 & SMOTE)
4,NPV,99.3473,98.3441,100.0000,Random Forest (VL diag + 1 & SMOTE)
5,Precision,20.4276,14.1961,28.1250,Random Forest (VL diag + 1 & SMOTE)
6,Sensitivity,80.9000,50.0000,100.0000,Random Forest (VL diag + 1 & SMOTE)
7,Specificity,89.7944,84.2601,93.6215,Random Forest (VL diag + 1 & SMOTE)


## Fitting data list with VL at diagnosis & VL 2-days after diagnosis

In [8]:
rf_add2_list = model_func_rf_tune(split_list_diag2_valid_smote_final)
rf_add2_met_summary = sum_metric(rf_add2_list)
rf_add2_metrics_summary = rf_add2_met_summary["metric_summary"]
rf_add2_summary = met_collate_func(rf_add2_metrics_summary).assign(
    models = "Random Forest (VL diag + 2 & SMOTE)"
)
rf_add2_summary

Bootstrapping pr_auc: 100%|██████████| 1000/1000 [00:00<00:00, 1569.84it/s]


bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,45.3322,22.2453,67.6982,Random Forest (VL diag + 2 & SMOTE)
1,AUROC_value,94.2847,87.2430,98.4899,Random Forest (VL diag + 2 & SMOTE)
2,Accuracy,89.8248,84.2749,93.3535,Random Forest (VL diag + 2 & SMOTE)
3,Accuracy_train,90.9979,88.2386,93.7967,Random Forest (VL diag + 2 & SMOTE)
4,NPV,99.3557,98.3813,100.0000,Random Forest (VL diag + 2 & SMOTE)
5,Precision,20.9718,13.9916,29.1094,Random Forest (VL diag + 2 & SMOTE)
6,Sensitivity,81.1000,50.0000,100.0000,Random Forest (VL diag + 2 & SMOTE)
7,Specificity,90.0966,83.9486,94.0810,Random Forest (VL diag + 2 & SMOTE)


# Save model trained

In [9]:
path = os.getcwd()

state = {
    "rf_fil_list": rf_fil_list,
    "rf_vlsymp_sim_list": rf_vlsymp_sim_list,
    "rf_vldiag_list": rf_vldiag_list,
    "rf_add1_list": rf_add1_list,
    "rf_add2_list": rf_add2_list
}

save_file = os.path.join(path, "rf_trained.pkl")

with open(save_file, "wb") as f:
    dill.dump(state, f)

#print(f"Saved to: {save_file}")

In [10]:
path = os.getcwd()

state = {
    "rf_fil_summary": rf_fil_summary,
    "rf_vldiag_summary": rf_vldiag_summary,
    "rf_add1_summary": rf_add1_summary,
    "rf_add2_summary": rf_add2_summary,
    "rf_vlsymp_sim_summary": rf_vlsymp_sim_summary
}

save_file = os.path.join(path, "rf_metric_summary.pkl")

with open(save_file, "wb") as f:
    dill.dump(state, f)

#print(f"Saved to: {save_file}")